# ASL CNN-LSTM Training Notebook

Train a research-grade ASL recognition model on the WLASL-25 dataset.

**Steps:**
1. Upload your processed data (X_train.npy, y_train.npy, etc.)
2. Run training
3. Export to TensorFlow.js
4. Download model for your app

In [ ]:
# Install dependencies
!pip install tensorflowjs -q

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers, Model
from google.colab import files

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 1. Upload Data

Upload these files from `scripts/lstm_training/data/processed/`:
- X_train.npy, y_train.npy
- X_val.npy, y_val.npy
- X_test.npy, y_test.npy
- metadata.json

In [ ]:
# Upload files
uploaded = files.upload()
print(f"\nUploaded: {list(uploaded.keys())}")

In [ ]:
# Load data
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

with open('metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")
print(f"Classes: {len(metadata['vocabulary'])}")
print(f"Vocabulary: {metadata['vocabulary']}")

## 2. Define Model

In [ ]:
class AttentionLayer(layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.W = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer='zeros', trainable=True)
        self.u = self.add_weight(shape=(self.units,), initializer='glorot_uniform', trainable=True)

    def call(self, inputs, mask=None):
        u_it = tf.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        a_it = tf.tensordot(u_it, self.u, axes=1)
        if mask is not None:
            mask = tf.cast(mask, dtype=a_it.dtype)
            a_it = a_it + (1 - mask) * (-1e9)
        alpha = tf.nn.softmax(a_it, axis=1)
        return tf.reduce_sum(inputs * tf.expand_dims(alpha, -1), axis=1)

    def get_config(self):
        return {**super().get_config(), 'units': self.units}


def create_cnn_lstm_model(num_classes, window_size=16, feature_count=63):
    inputs = layers.Input(shape=(window_size, feature_count))
    x = layers.Masking(mask_value=0.0)(inputs)
    x = layers.Reshape((window_size, feature_count, 1))(x)
    x = layers.TimeDistributed(layers.Conv1D(128, 3, activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.Dropout(0.3)(x)
    x = layers.TimeDistributed(layers.Flatten())(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = AttentionLayer(units=128)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs, name='cnn_lstm')


num_classes = len(metadata['vocabulary'])
model = create_cnn_lstm_model(num_classes)
model.summary()

## 3. Train Model

In [ ]:
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
)

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.legend()

plt.show()

## 4. Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Best Val Accuracy: {max(history.history['val_accuracy']):.4f}")

## 5. Export to TensorFlow.js

In [ ]:
import tensorflowjs as tfjs
import shutil

# Save Keras model
model.save('asl_cnn_lstm_25.keras')

# Export to TF.js
tfjs.converters.save_keras_model(model, 'tfjs_model')

# Create zip for download
shutil.make_archive('asl_cnn_lstm_25_tfjs', 'zip', 'tfjs_model')

print("✓ Model exported to TensorFlow.js format")
print("\nFiles in tfjs_model/:")
!ls -la tfjs_model/

In [ ]:
# Download the model
files.download('asl_cnn_lstm_25_tfjs.zip')

print("\n" + "="*60)
print("NEXT STEPS:")
print("="*60)
print("1. Unzip asl_cnn_lstm_25_tfjs.zip")
print("2. Copy contents to public/models/asl_cnn_lstm_25/")
print("3. Rename model.json to asl_cnn_lstm_25.json in public/models/")
print("4. Update weight paths in asl_cnn_lstm_25.json to include subfolder")